# 05 Multimodal Model: Stacking (Late Fusion)

Detects manipulated news image-caption pairs using the image, the caption **and how well they fit together**. Two outputs, as in the project plan:
- `image_fake`: was the image manipulated (face_swap / face_attribute)?
- `text_fake`: was the caption manipulated (text_swap / text_attribute)?

A pair is flagged as fake (`is_fake`) if either output is high.

**Architecture: a two-level stack**

| Level | Model | Input | Output |
|---|---|---|---|
| 1 | Logistic regression **per face** | CLIP embedding of each face crop | P(this face was edited), summarised over the image's faces |
| 1 | Logistic regression | CLIP image embedding | P(image manipulated) |
| 1 | Logistic regression | CLIP caption embedding | P(caption manipulated) |
| 1 | Logistic regression | CLIP image + caption embeddings and their element-wise product | P(caption manipulated), using image-caption agreement |
| 2 | Gradient boosting, one per target | Level-1 scores + hand-crafted text and image features (`03`) + CLIP similarities (`04`) | `image_fake`, `text_fake` |

Why this design:
- **CLIP embeddings + linear classifier:** frozen CLIP ViT-L/14 features with a linear classifier are a strong, cheap detector of generated images (Ojha et al., 2023, *Towards Universal Fake Image Detectors*), and CLIP's shared image-text space directly measures whether a caption fits its photo.
- **Face level:** in training we know which face was edited (`fake_image_box`), so the face classifier learns from the edited face itself rather than the whole photo. At prediction time it only uses the face boxes and pixels; the image score is the most suspicious face.
- **Stacking:** level 2 combines signals of very different kinds (768-dimensional embeddings vs. a handful of forensic statistics), and gradient boosting handles non-linear interactions such as "low image-caption similarity *and* strong sentiment".
- **Leakage control:** level-1 scores for training rows are **out-of-fold** (5-fold, grouped by article `id`), so level 2 never sees a score from a model that was trained on that row.

**Data use:** models are fit on `train_sample`, compared and tuned on `validation`. `test` is evaluated only in the last section, behind `RUN_TEST`, once the final model is chosen.

## 0. Setup

In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import (best_f1_threshold, combine_fake_score, detection_rate_by_type, evaluate,
                            prevalence_table)
from src.feature_sets import ARTIFACT_FEATURES, feature_columns

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "data" / "features"
CLIP_CACHE = PROJECT_ROOT / "data" / "interim" / "clip"
PREDICTIONS_DIR = PROJECT_ROOT / "results" / "predictions"
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 4012
N_FOLDS = 5
TARGETS = ["image_fake", "text_fake"]

# Leave False while developing. Set True once, at the end, to report final test results.
RUN_TEST = False

sns.set_theme(style="whitegrid")

In [ ]:
SPLITS = ["train_sample", "validation", "test"]
FEATURE_GROUPS = ["text", "image", "clip", "clip_face", "clip_emb"]


def load_split(split):
    df = pd.read_parquet(PROCESSED_DIR / f"{split}.parquet")
    for group in FEATURE_GROUPS:
        df = df.merge(pd.read_parquet(FEATURES_DIR / f"{group}_{split}.parquet"), on="row_id", how="left")
    assert df.filter(regex="^(txt|img|clip)_").notna().all().all(), f"missing features in {split}"
    return df


data = {split: load_split(split) for split in SPLITS}
train, val = data["train_sample"], data["validation"]

IMG_EMB = [c for c in train.columns if c.startswith("clip_img_")]
TXT_EMB = [c for c in train.columns if c.startswith("clip_txt_")]
print({split: df.shape for split, df in data.items()})
print(f"CLIP embedding size: {len(IMG_EMB)} (image) + {len(TXT_EMB)} (caption)")

## 1. Level 1: Linear Classifiers on CLIP Embeddings

`oof_probe` fits a scaled logistic regression and returns:
- out-of-fold scores for the training rows (each fold's model never saw that row's article)
- scores for validation and test from a model fit on all training rows

In [ ]:
def make_probe(C):
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000))


def oof_probe(X_train, y_train, groups, X_others, C):
    """Out-of-fold scores on train + scores on each array in X_others."""
    oof = np.zeros(len(X_train))
    for fit_idx, pred_idx in GroupKFold(n_splits=N_FOLDS).split(X_train, y_train, groups):
        model = make_probe(C).fit(X_train[fit_idx], y_train[fit_idx])
        oof[pred_idx] = model.predict_proba(X_train[pred_idx])[:, 1]
    full = make_probe(C).fit(X_train, y_train)
    return oof, [full.predict_proba(X)[:, 1] for X in X_others]


def choose_C(X_train, y_train, groups, candidates=(0.001, 0.01, 0.1)):
    """Pick the regularisation strength by grouped CV AUC on train (validation stays untouched)."""
    fit_idx, pred_idx = next(GroupKFold(n_splits=N_FOLDS).split(X_train, y_train, groups))
    aucs = {}
    for C in candidates:
        model = make_probe(C).fit(X_train[fit_idx], y_train[fit_idx])
        aucs[C] = roc_auc_score(y_train[pred_idx], model.predict_proba(X_train[pred_idx])[:, 1])
    print("  CV AUC by C:", {c: round(a, 4) for c, a in aucs.items()})
    return max(aucs, key=aucs.get)

### 1.1 Face-Level Classifier

Face embeddings come from the cache written by `04` (`data/interim/clip/faces.npy`), one row per face in the same order `04` produced them. Each training face is labelled 1 if it is the face in `fake_image_box`, else 0. The per-image score is summarised as the most suspicious face (`max`), the average, and the largest face.

In [ ]:
# Rebuild the face order used by 04: distinct images in (train_sample, validation, test) order, faces in box order
all_rows = pd.concat([data[s] for s in SPLITS], ignore_index=True)
images = all_rows.drop_duplicates("image")[["image", "id", "mtcnn_boxes"]].reset_index(drop=True)
n_faces = images["mtcnn_boxes"].str.len().to_numpy()

face_emb = np.load(CLIP_CACHE / "faces.npy")
assert len(face_emb) == n_faces.sum(), "faces.npy does not match the face order; re-run 04"

faces = pd.DataFrame({
    "image": np.repeat(images["image"].to_numpy(), n_faces),
    "id": np.repeat(images["id"].to_numpy(), n_faces),
    "box": [list(map(int, b)) for boxes in images["mtcnn_boxes"] for b in boxes],
})
faces["area"] = faces["box"].apply(lambda b: (b[2] - b[0]) * (b[3] - b[1]))

# Face label: is this the manipulated face? Taken from fake_image_box of the rows that use this image
fake_box = (
    all_rows[all_rows["image_fake"] == 1].drop_duplicates("image").set_index("image")["fake_image_box"]
    .apply(lambda b: list(map(int, b)))
)
faces["face_fake"] = [int(fake_box.get(img) == box) for img, box in zip(faces["image"], faces["box"])]

image_split = all_rows.drop_duplicates("image").set_index("image")["row_id"].str.rsplit("_", n=1).str[0]
faces["split"] = faces["image"].map(image_split)
print(faces.groupby("split")["face_fake"].agg(["size", "sum"]).rename(columns={"size": "faces", "sum": "edited faces"}))

In [ ]:
start = time.time()
is_train_face = (faces["split"] == "train").to_numpy()
X_face_train, y_face_train = face_emb[is_train_face], faces.loc[is_train_face, "face_fake"].to_numpy()
face_groups = faces.loc[is_train_face, "id"].to_numpy()

C_face = choose_C(X_face_train, y_face_train, face_groups)
oof_face, (other_face,) = oof_probe(X_face_train, y_face_train, face_groups, [face_emb[~is_train_face]], C_face)

faces["p_face"] = 0.0
faces.loc[is_train_face, "p_face"] = oof_face
faces.loc[~is_train_face, "p_face"] = other_face
print(f"face classifier (C={C_face}): {time.time() - start:.0f}s | "
      f"out-of-fold face AUC {roc_auc_score(y_face_train, oof_face):.3f}")

# Per-image summaries, then attach to every row that uses the image
per_image = faces.groupby("image").agg(
    l1_face_max=("p_face", "max"),
    l1_face_mean=("p_face", "mean"),
)
per_image["l1_face_largest"] = faces.loc[faces.groupby("image")["area"].idxmax()].set_index("image")["p_face"]
for split in SPLITS:
    data[split] = data[split].drop(columns=per_image.columns, errors="ignore").merge(
        per_image, left_on="image", right_index=True, how="left")
train, val = data["train_sample"], data["validation"]

### 1.2 Image, Caption and Pair Classifiers

- `l1_image`: whole-image embedding → `image_fake`
- `l1_caption`: caption embedding → `text_fake` (can pick up sentiment-flipped wording)
- `l1_pair`: image embedding, caption embedding and their element-wise product → `text_fake`. The product lets a linear model see *agreement* between image and caption dimension by dimension, which a single similarity score compresses into one number.

In [ ]:
def embedding_matrix(df, kind):
    img = df[IMG_EMB].to_numpy(np.float32)
    txt = df[TXT_EMB].to_numpy(np.float32)
    return {"image": img, "caption": txt, "pair": np.hstack([img, txt, img * txt])}[kind]


LEVEL1 = {"l1_image": ("image", "image_fake"), "l1_caption": ("caption", "text_fake"), "l1_pair": ("pair", "text_fake")}
groups = train["id"].to_numpy()

for name, (kind, target) in LEVEL1.items():
    start = time.time()
    X_train = embedding_matrix(train, kind)
    y_train = train[target].to_numpy()
    print(name)
    C = choose_C(X_train, y_train, groups)
    oof, others = oof_probe(X_train, y_train, groups, [embedding_matrix(data[s], kind) for s in SPLITS[1:]], C)
    data["train_sample"][name] = oof
    for split, pred in zip(SPLITS[1:], others):
        data[split][name] = pred
    print(f"  {time.time() - start:.0f}s | out-of-fold AUC vs {target}: {roc_auc_score(y_train, oof):.3f}")

train, val = data["train_sample"], data["validation"]

In [ ]:
L1 = ["l1_face_max", "l1_face_mean", "l1_face_largest", "l1_image", "l1_caption", "l1_pair"]
l1_auc = pd.DataFrame({
    target: {c: roc_auc_score(val[target], val[c]) for c in L1} for target in ["image_fake", "text_fake", "is_fake"]
}).round(3)
l1_auc.columns = [f"val AUC vs {t}" for t in l1_auc.columns]
l1_auc

## 2. Level 2: Gradient Boosting per Target

Feature sets (artifact features from `src/feature_sets.py` are excluded unless stated):
- **TEXT**: hand-crafted caption features (`txt_*`)
- **IMAGE**: hand-crafted forensic features (`img_*`)
- **CROSS**: CLIP image-caption and face-caption similarities (`clip_sim*`, `clip_face_*`)
- **LEVEL1**: the level-1 scores above

In [ ]:
TEXT = feature_columns(train, "txt_")
IMAGE = feature_columns(train, "img_")
CROSS = [c for c in train.columns if c.startswith("clip_sim") or c.startswith("clip_face_")]
ALL = TEXT + IMAGE + CROSS + L1
print({"TEXT": len(TEXT), "IMAGE": len(IMAGE), "CROSS": len(CROSS), "LEVEL1": len(L1), "ALL": len(ALL)})


def make_gbm():
    return HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=600, max_leaf_nodes=31, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=30, random_state=RANDOM_STATE,
    )


def fit_level2(features):
    """Fit one model per target on train_sample. Returns models and validation probabilities."""
    models, preds = {}, {}
    for target in TARGETS:
        models[target] = make_gbm().fit(train[features], train[target])
        preds[target] = models[target].predict_proba(val[features])[:, 1]
    return models, preds


start = time.time()
models, val_pred = fit_level2(ALL)
print(f"level 2 fit: {time.time() - start:.0f}s")

### 2.1 Thresholds and Validation Results

Each output's threshold is set to maximise F1 on validation. AUC, PR-AUC and EER do not depend on the threshold.

In [ ]:
val_scores = {"image_fake": val_pred["image_fake"], "text_fake": val_pred["text_fake"],
              "is_fake": combine_fake_score(val_pred["image_fake"], val_pred["text_fake"])}
thresholds = {target: best_f1_threshold(val[target], s) for target, s in val_scores.items()}

results_val = evaluate(val, val_pred["image_fake"], val_pred["text_fake"], thresholds)
results_val.round(3)

## 3. Baselines and Ablations

**Comparison models** (all on validation):
- Logistic regression on the same level-2 features: is the non-linear model needed?
- Gradient boosting on each feature family alone and in combination: what does each modality contribute, and does the **cross-modal** information add anything beyond text and image features?
- With artifact features: how much would the model gain from the dataset shortcuts found in `03`? (A gain here is *not* a real improvement.)

In [ ]:
def auc_row(pred):
    s = {"image_fake": pred["image_fake"], "text_fake": pred["text_fake"],
         "is_fake": combine_fake_score(pred["image_fake"], pred["text_fake"])}
    return {f"AUC {t}": roc_auc_score(val[t], s[t]) for t in ["is_fake", "image_fake", "text_fake"]}


def logreg_level2(features):
    preds = {}
    for target in TARGETS:
        m = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)).fit(train[features], train[target])
        preds[target] = m.predict_proba(val[features])[:, 1]
    return preds


ARTIFACTS = [c for c in ARTIFACT_FEATURES if c in train.columns]
experiments = {
    "Text features only": TEXT,
    "Image features only": IMAGE,
    "Text + image (no CLIP)": TEXT + IMAGE,
    "Cross-modal similarities only": CROSS,
    "Level-1 CLIP scores only": L1,
    "All without cross-modal": TEXT + IMAGE + ["l1_face_max", "l1_face_mean", "l1_face_largest", "l1_image", "l1_caption"],
    "All (final)": ALL,
    "All + artifact features": ALL + ARTIFACTS,
}

ablation = {}
start = time.time()
for name, features in experiments.items():
    ablation[name] = auc_row(val_pred if name == "All (final)" else fit_level2(features)[1])
ablation["Logistic regression, all features"] = auc_row(logreg_level2(ALL))
print(f"{time.time() - start:.0f}s")

ablation = pd.DataFrame(ablation).T.round(3)
ablation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = ablation.drop(index=["All + artifact features"]).reset_index(names="model").melt(
    id_vars="model", var_name="metric", value_name="AUC")
sns.barplot(data=plot_df, y="model", x="AUC", hue="metric", ax=ax)
ax.set_xlim(0.5, 1.0)
ax.set_title("Validation AUC by Feature Set")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 4. Error Analysis on Validation

In [ ]:
# Detection rate per manipulation class at the chosen is_fake threshold.
# For "orig" this is the false-positive rate.
detection = detection_rate_by_type(val, val_scores["is_fake"], thresholds["is_fake"])
detection.round(3)

In [ ]:
# Does each output catch the manipulation it is meant for?
per_output = pd.DataFrame({
    "image output flags": pd.Series(val_scores["image_fake"] >= thresholds["image_fake"], index=val.index),
    "text output flags": pd.Series(val_scores["text_fake"] >= thresholds["text_fake"], index=val.index),
}).groupby(val["fake_cls"]).mean().round(3)
per_output

In [ ]:
# Which features does each level-2 model rely on? Permutation importance: AUC drop on validation when a feature is shuffled.
importance = {}
sample = val.sample(8000, random_state=RANDOM_STATE)
for target in TARGETS:
    r = permutation_importance(models[target], sample[ALL], sample[target], scoring="roc_auc",
                               n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1)
    importance[target] = pd.Series(r.importances_mean, index=ALL)
importance = pd.DataFrame(importance)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, target in zip(axes, TARGETS):
    importance[target].sort_values().tail(12).plot.barh(ax=ax)
    ax.set_title(f"Top features for {target}")
    ax.set_xlabel("AUC drop when shuffled")
plt.tight_layout()
plt.show()

## 5. Realistic Prevalence

Validation is 66% fake. At realistic fake rates, the same recall and false-positive rate produce very different alert queues. Using the `is_fake` threshold chosen above, per 100,000 posts:

In [ ]:
prevalence_table(val["is_fake"], val_scores["is_fake"], thresholds["is_fake"]).round(3)

At low fake rates, false alarms dominate. In deployment the threshold would be raised to fit reviewer capacity; the curve below shows the trade-off at a 1% fake rate.

In [ ]:
grid = np.quantile(val_scores["is_fake"], np.linspace(0.30, 0.995, 60))
curve = pd.concat([prevalence_table(val["is_fake"], val_scores["is_fake"], t, prevalences=[0.01]).assign(threshold=t)
                   for t in grid])

fig, ax1 = plt.subplots(figsize=(9, 4.5))
ax1.plot(curve["alerts"], curve["recall"], label="recall (share of fakes caught)")
ax1.plot(curve["alerts"], curve["precision"], label="precision (share of alerts that are fake)")
ax1.set_xscale("log")
ax1.set_xlabel("Alerts per 100,000 posts (log scale)")
ax1.set_ylabel("Rate")
ax1.set_title("Reviewer Workload vs Detection at a 1% Fake Rate (validation)")
ax1.legend()
plt.tight_layout()
plt.show()

## 6. Save Validation Predictions

In [ ]:
def predictions_frame(df, pred):
    return pd.DataFrame({
        "row_id": df["row_id"].to_numpy(),
        "p_image_fake": pred["image_fake"],
        "p_text_fake": pred["text_fake"],
        "p_is_fake": combine_fake_score(pred["image_fake"], pred["text_fake"]),
    })


predictions_frame(val, val_pred).to_parquet(PREDICTIONS_DIR / "multimodal_stacking_validation.parquet", index=False)
pd.Series(thresholds, name="threshold").to_frame().to_csv(PREDICTIONS_DIR / "multimodal_stacking_thresholds.csv")
print("saved to", PREDICTIONS_DIR.relative_to(PROJECT_ROOT))

## 7. Test Evaluation (run once, at the end)

Uses the models and thresholds fixed on validation. Keep `RUN_TEST = False` until the model is final.

In [ ]:
if RUN_TEST:
    test = data["test"]
    test_pred = {target: models[target].predict_proba(test[ALL])[:, 1] for target in TARGETS}
    display(evaluate(test, test_pred["image_fake"], test_pred["text_fake"], thresholds).round(3))
    test_is_fake = combine_fake_score(test_pred["image_fake"], test_pred["text_fake"])
    display(detection_rate_by_type(test, test_is_fake, thresholds["is_fake"]).round(3))
    display(prevalence_table(test["is_fake"], test_is_fake, thresholds["is_fake"]).round(3))
    predictions_frame(test, test_pred).to_parquet(PREDICTIONS_DIR / "multimodal_stacking_test.parquet", index=False)
else:
    print("RUN_TEST is False: test set not evaluated.")

# Summary

Fill in after running.